# Pipeline de Reconocimiento Facial

## Descripción General del Pipeline

Todo en este módulo converge aquí. El pipeline de reconocimiento facial encadena cuatro conceptos separados:

1. **Detección** (YOLO) — localizar dónde aparece una persona en una imagen.
2. **Embedding** (CLIP) — convertir el recorte de cara detectado en un vector de 512 dimensiones.
3. **Almacenamiento** (ChromaDB) — persistir ese vector junto con metadatos de identidad.
4. **Verificación** — dada una nueva imagen de cara, encontrar el embedding almacenado más cercano y decidir si la similitud es suficientemente alta para confirmar la identidad.

Esta es la arquitectura que subyace a prácticamente todos los sistemas de reconocimiento facial desplegados, desde el control de acceso de puertas hasta el desbloqueo móvil. Los componentes difieren en escala y precisión, pero la estructura del pipeline es la misma.

El pipeline completo tiene cinco pasos:

| Paso | Qué ocurre | Herramienta clave |
|------|-------------|---------|
| **1. Captura** | Obtener una foto de una cara (webcam o archivo) | OpenCV `VideoCapture` |
| **2. Detectar** | Localizar la cara dentro de la imagen, recortarla | YOLO `yolov8n.pt` |
| **3. Embedir** | Convertir el recorte de cara en un vector de 512 dimensiones | CLIP `clip-ViT-B-32` |
| **4. Almacenar** | Guardar el vector + metadatos en una BD | ChromaDB |
| **5. Verificar** | Consultar la BD con una nueva cara; decidir coincidencia/no-coincidencia | similitud coseno |

**Inscripción** = pasos 1-4 (se hace una vez, al registrar a una persona).
**Verificación** = pasos 1-3 y luego 5 (se hace en cada intento de acceso).

## Paso 1: Capturar una Cara

En un sistema de producción, los fotogramas provienen de una webcam en vivo.
Aquí cargamos imágenes de retrato estáticas desde la carpeta `resources/images/` para mantener el notebook
completamente ejecutable sin conexión, y luego mostramos el código equivalente para webcam con anotaciones detalladas.

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

IMG_DIR = Path('./resources/images')
names = ['scene1', 'scene2', 'scene3']
face_images = {name: cv2.imread(str(IMG_DIR / f'{name}.jpg')) for name in names}
for name, img in face_images.items():
    assert img is not None, f'No se pudo cargar {IMG_DIR / name}.jpg'
    print(f'{name}: {img.shape}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, name in zip(axes, names):
    ax.imshow(cv2.cvtColor(face_images[name], cv2.COLOR_BGR2RGB))
    ax.set_title(name)
    ax.axis('off')
plt.suptitle('Fotos de retrato usadas como sustitutos de capturas reales de webcam')
plt.tight_layout()
plt.show()

### Código de Captura por Webcam

> **Nota:** La celda siguiente se muestra como referencia — requiere una webcam física y
> una pantalla con interfaz gráfica. No la ejecutes en un entorno sin cabeza/nube.

En el proyecto completo esta lógica reside en `authentication.py → capture_student_authentication()`.

In [ ]:
# ⚠️ SOLO REFERENCIA — requiere webcam + pantalla con GUI
# Este es el patrón usado por computer-vision/implementation/src/authentication.py
def capture_faces_from_webcam(n_captures: int = 5) -> list[np.ndarray]:
    captures = []
    cap = cv2.VideoCapture(0)  # 0 = primera cámara
    while len(captures) < n_captures:
        ret, frame = cap.read()
        if not ret:
            break
        # Control de calidad 1: brillo
        # Un valor de píxel medio fuera de 50–200 significa demasiado oscuro o sobreexpuesto.
        brightness = frame.mean()
        if not (50 < brightness < 200):
            cv2.putText(frame, 'Bad lighting!', (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            cv2.imshow('Capture', frame)
            cv2.waitKey(100)
            continue
        # Control de calidad 2: desenfoque (varianza del Laplaciano)
        # Un fotograma borroso tiene poca variación en la segunda derivada.
        # El umbral de 100 es una regla empírica habitual.
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
        if blur_score < 100:
            cv2.putText(frame, 'Too blurry!', (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            cv2.imshow('Capture', frame)
            cv2.waitKey(100)
            continue
        captures.append(frame.copy())
        cv2.putText(frame, f'Captured {len(captures)}/{n_captures}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow('Capture', frame)
        cv2.waitKey(500)  # pausa breve entre capturas
    # Libera la cámara para que otras aplicaciones puedan usarla.
    cap.release()
    # Cierra todas las ventanas GUI creadas por cv2.imshow().
    cv2.destroyAllWindows()
    return captures

# _ = capture_faces_from_webcam()

## Paso 2: Detectar la Cara con YOLO

El modelo general de YOLO (`yolov8n.pt`) detecta personas como **clase 0**.
Tomamos el recuadro de persona con mayor confianza y lo recortamos como la región facial.
Para un despliegue real, usa `yolov8n-face.pt`, que está ajustado específicamente
para caras y devuelve recuadros más precisos alrededor de la cara en lugar del cuerpo completo.

In [ ]:
from ultralytics import YOLO

# Cargar el modelo nano (el más pequeño/rápido).
# Descarga ~6 MB en la primera ejecución.
detector = YOLO('yolov8n.pt')
print('Detector cargado')

In [ ]:
def detect_and_crop_face(img: np.ndarray, padding: int = 20) -> np.ndarray:
    h, w = img.shape[:2]
    # Ejecutar inferencia; suprimir la salida en consola con verbose=False.
    results = detector(img, verbose=False)
    boxes = results[0].boxes
    # Filtrar a la clase 0 (persona), quedarse con la detección de mayor confianza.
    # En fotos sin persona detectada, usar la imagen completa como alternativa.
    best_box = None
    best_conf = 0
    for box in boxes:
        if int(box.cls[0]) == 0 and float(box.conf[0]) > best_conf:
            best_conf = float(box.conf[0])
            best_box = box.xyxy[0].cpu().numpy().astype(int)
    if best_box is None:
        # No se detectó ninguna persona — usar la imagen completa como 'cara'.
        # En un sistema de producción se rechazaría este fotograma.
        return img
    x1, y1, x2, y2 = best_box
    # Añadir relleno alrededor del recuadro, limitado a los bordes de la imagen.
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(w, x2 + padding)
    y2 = min(h, y2 + padding)
    return img[y1:y2, x1:x2]

face_crops = {name: detect_and_crop_face(face_images[name]) for name in names}
for name, crop in face_crops.items():
    print(f'{name}: crop shape = {crop.shape}')

El parámetro `padding` amplía el recuadro delimitador `padding` píxeles en todos los lados antes del recorte. Esto tiene dos propósitos:

1. **Incluir contexto**: YOLO devuelve recuadros de personas, que pueden recortar ajustadamente la cara. Un pequeño relleno asegura que la frente y el mentón no se corten.
2. **Mejorar la precisión de CLIP**: CLIP fue entrenado con imágenes que tienen contexto natural alrededor de los objetos, no recortes perfectamente ajustados. Dejar algo de fondo generalmente mejora la calidad del embedding.

El recuadro se limita a los bordes de la imagen, por lo que el relleno nunca lee fuera de la imagen.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, name in zip(axes, names):
    ax.imshow(cv2.cvtColor(face_crops[name], cv2.COLOR_BGR2RGB))
    ax.set_title(f'{name} (recorte)')
    ax.axis('off')
plt.suptitle('Paso 2: Recortes de cara (se usa la imagen completa cuando YOLO no detecta ninguna persona)')
plt.tight_layout()
plt.show()

## Paso 3: Extraer un Embedding de Características con CLIP

Pasamos cada recorte de cara a través del codificador de imágenes CLIP para obtener un vector de 512 dimensiones.
Dos detalles de implementación son importantes aquí:

- **Normalización L2 antes del almacenamiento**: Los embeddings de CLIP *no* son automáticamente de norma unitaria. Los normalizamos para que el producto escalar sea igual a la similitud coseno, y para que el índice de distancia coseno de ChromaDB dé resultados correctos. Esto coincide con lo que mostró el notebook de embeddings independiente: los vectores normalizados permiten que las comparaciones de distancia se centren en la dirección (identidad) en lugar de la magnitud (que no tiene significado aquí).
- **Conversión BGR → RGB**: OpenCV carga imágenes en BGR; CLIP (vía PIL) espera RGB. Omitir este paso produciría embeddings ligeramente diferentes, pero más importante aún, corrompería silenciosamente las características dependientes del color.

In [ ]:
from sentence_transformers import SentenceTransformer

clip_model = SentenceTransformer('clip-ViT-B-32')
print(f'Dimensión del embedding: {clip_model.get_sentence_embedding_dimension()}')

In [ ]:
def embed_face(bgr_crop: np.ndarray) -> np.ndarray:
    # Convertir BGR (por defecto en OpenCV) → RGB (por defecto en PIL/CLIP).
    rgb = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb)
    emb = clip_model.encode(pil_img)   # array numpy float32, forma (512,)
    # Normalización L2 para que el producto escalar == similitud coseno.
    emb = emb / np.linalg.norm(emb)
    return emb

for name in names:
    emb = embed_face(face_crops[name])
    print(f'{name}: shape={emb.shape}  norm={np.linalg.norm(emb):.6f}  (debería ser ~1.0 tras la normalización)')

## Paso 4: Almacenar Embeddings en ChromaDB

Creamos una colección persistente en ChromaDB e insertamos un embedding por estudiante.
Los metadatos incluyen el nombre del estudiante y la marca de tiempo de la inscripción.

In [ ]:
import chromadb
import datetime
from pathlib import Path

# PersistentClient guarda los datos en disco — sobrevive a los reinicios del notebook.
db_path = Path('./artifacts/face_db')
db_path.mkdir(parents=True, exist_ok=True)
db_client = chromadb.PersistentClient(path=str(db_path))

# Eliminar la colección si ya existe (pizarra limpia para esta demo).
try:
    db_client.delete_collection('enrolled_faces')
    print('Colección antigua eliminada')
except Exception:
    pass

# La distancia coseno es la métrica correcta para embeddings de CLIP normalizados en L2.
face_collection = db_client.create_collection(
    name='enrolled_faces',
    metadata={'hnsw:space': 'cosine'})
print('Colección lista')

In [ ]:
# Inscribir los tres estudiantes
for name in names:
    emb = embed_face(face_crops[name])
    face_collection.add(
        ids        = [f'{name}_enroll_001'],
        embeddings = [emb.tolist()],
        metadatas  = [{'student': name,
                        'enrolled_at': datetime.datetime.now().isoformat(),
                        'source': 'static_image_demo'}]
    )
print(f'Inscritas {face_collection.count()} cara(s) en ChromaDB')

## Paso 5: Verificar Identidad (Búsqueda por Similitud)

En el momento de la verificación, el estudiante presenta una nueva foto.
Lo embebemos y consultamos la base de datos para el embedding almacenado más cercano.

**Regla de decisión**: si la coincidencia más alta tiene cosine_similarity ≥ umbral → identidad confirmada.

Ajuste del umbral:
- Demasiado alto (p. ej. 0.99) → muchos falsos rechazos (orientado a la seguridad)
- Demasiado bajo (p. ej. 0.5) → fácil de suplantar con cualquier persona similar (orientado a la comodidad)
- El proyecto usa **0.85** como valor predeterminado equilibrado para autenticación académica.

In [ ]:
VERIFICATION_THRESHOLD = 0.85

def verify_identity(query_crop: np.ndarray, student_name: str | None = None) -> dict:
    query_emb = embed_face(query_crop)
    # Opcionalmente limitar la búsqueda a un estudiante específico (para verificación 1:1).
    where_clause = {'student': student_name} if student_name else None
    results = face_collection.query(
        query_embeddings = [query_emb.tolist()],
        n_results = 3,
        where = where_clause,
        include = ['metadatas', 'distances'],
    )
    top_id   = results['ids'][0][0]
    top_dist = results['distances'][0][0]
    top_meta = results['metadatas'][0][0]
    # ChromaDB devuelve DISTANCIA coseno (0 = idéntico, 2 = opuesto).
    # Convertir a similitud: similarity = 1 − distance.
    similarity = 1.0 - top_dist
    return {
        'verified': similarity >= VERIFICATION_THRESHOLD,
        'similarity': similarity,
        'matched_id': top_id,
        'matched_student': top_meta['student'],
        'all_results': list(zip(results['ids'][0], [1-d for d in results['distances'][0]]))
    }

## Demo de Inscripción + Verificación

Consultamos la imagen inscrita de cada estudiante (se espera coincidencia),
luego consultamos la imagen de student_a pero preguntando por student_b (se espera no-coincidencia).

In [ ]:
print('=== Consultas de la misma persona (deberían verificar) ===')
for name in names:
    result = verify_identity(face_crops[name])
    status = '✓ VERIFICADO' if result['verified'] else '✗ RECHAZADO'
    print(f'{name} → coincidencia={result["matched_student"]:12s}  similarity={result["similarity"]:.4f}  {status}')

print()
print('=== Consulta entre personas (debería rechazar) ===')
result = verify_identity(face_crops[names[0]], student_name=names[1])
status = '✓ VERIFICADO' if result['verified'] else '✗ RECHAZADO (correcto!)'
print(f'student_a consultado contra student_b → similarity={result["similarity"]:.4f}  {status}')

In [ ]:
# Visualizar similitudes como mapa de calor
n = len(names)
sim_matrix = np.zeros((n, n))
for i, query_name in enumerate(names):
    q_emb = embed_face(face_crops[query_name])
    for j, stored_name in enumerate(names):
        results = face_collection.query(
            query_embeddings=[q_emb.tolist()],
            n_results=1,
            where={'student': stored_name},
            include=['distances']
        )
        sim_matrix[i, j] = 1 - results['distances'][0][0]

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(sim_matrix, vmin=0, vmax=1, cmap='RdYlGn')
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_yticklabels(names)
ax.set_xlabel('Almacenado (inscrito)')
ax.set_ylabel('Consulta')
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i,j]:.3f}', ha='center', va='center', fontsize=10,
                color='white' if sim_matrix[i,j] < 0.4 or sim_matrix[i,j] > 0.85 else 'black')
ax.axhline(0.5, color='grey', lw=0.5, ls='--')
ax.set_title('Similitud coseno consulta-vs-almacenado\n(diagonal = misma persona)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Conexión con el Proyecto Completo

Cada paso anterior se mapea directamente a un módulo en `computer-vision/implementation/src/`:

| Paso | Este notebook | Módulo del proyecto |
|------|--------------|----------------|
| Captura | `capture_faces_from_webcam()` | `authentication.py` -> `capture_student_authentication()` |
| Detectar + recortar | `detect_and_crop_face()` usando `yolov8n.pt` | `face_detector.py` -> `detect_faces()` |
| Embedir | `embed_face()` usando `clip-ViT-B-32` | `feature_extractor.py` -> `extract_features()` |
| Almacenar | `face_collection.add()` | `vector_database.py` -> `add_face()` |
| Verificar | `face_collection.query()` + umbral | `similarity_search.py` -> `search_by_face_image()` |

El proyecto añade características de producción que este notebook omite:
- **Múltiples capturas de inscripción** por estudiante (5-10 fotogramas) con controles de calidad
- **Configuración estructurada** vía `config/config.yaml` (umbrales, rutas de modelos, etc.)
- **Informes de autenticación JSON** y recortes de caras guardados para auditoría
- Un `main.py` de nivel superior que conecta todos los módulos

Un siguiente proyecto natural es reemplazar las imágenes de demo estáticas con tus propias capturas de webcam y construir un módulo de autenticación simple para ti mismo. Ese es el punto donde todo el módulo se une: detección, embeddings, búsqueda vectorial y umbrales de decisión en un solo flujo de trabajo.

## Limpieza

In [ ]:
import shutil
# Eliminar la base de datos de la demo
shutil.rmtree('artifacts/face_db', ignore_errors=True)
print('Limpieza completada')